# Leakage-Aware Evaluation of ML Models for Voice-Based Parkinson's Screening

Reproduces every result in the manuscript. Both datasets download automatically;
nothing needs to be uploaded.

**Runtime:** about 25 minutes on a standard Colab CPU instance. No GPU needed.

Run the cells in order. Section 6 measures timings **on this machine** and prints
a LaTeX block ready to paste into the paper.

## 1. Environment setup

In [1]:
!pip install -q catboost xgboost lightgbm shap
!apt-get -qq install -y unrar-free > /dev/null
print("dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.1 MB/s eta 0:00:00
dependencies installed


In [2]:
# Pin threads before heavy imports so timings are hardware-comparable.
import os
for v in ("OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS",
          "NUMEXPR_NUM_THREADS","VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

!git clone https://github.com/twki69/pd-screening-reproduction.git
%cd pd-screening-reproduction

# Option B: if you uploaded the pdpipe/ folder to this session, just continue.
import sys; sys.path.insert(0, ".")
print("ready")

Cloning into 'pd-screening-reproduction'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 38 (delta 12), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 32.60 KiB | 1.21 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/pd-screening-reproduction
ready


## 2. Load both cohorts

The Oxford cohort is the training benchmark; the Istanbul cohort is used only
for external validation. Note the composition: 32 subject identifiers, of which
just 8 are controls. That imbalance drives much of what follows.

In [3]:
import warnings; warnings.filterwarnings("ignore")
from pdpipe.data import load_oxford, load_istanbul, harmonised_oxford, load_oxford_meta

ox, ist = load_oxford(), load_istanbul()
print("Oxford  :", ox.summary())
print("Istanbul:", ist.summary())

Oxford  : {'cohort': 'Oxford', 'recordings': 195, 'subjects': 32, 'pd_subjects': 24, 'hc_subjects': 8, 'pd_recordings': 147, 'hc_recordings': 48, 'features': 22}
Istanbul: {'cohort': 'Istanbul', 'recordings': 756, 'subjects': 252, 'pd_subjects': 188, 'hc_subjects': 64, 'pd_recordings': 564, 'hc_recordings': 192, 'features': 16}


### Subject parsing and the 31-vs-32 discrepancy

Every filename matches `phon_<session>_<subject>_<index>`. The documentation
reports 31 participants; the filenames yield 32 codes. The 8 control codes match
the documentation exactly, so the discrepancy sits entirely among patients.

In [4]:
meta = load_oxford_meta()
g = meta.groupby("subject").agg(n=("filename","size"),
                                status=("status","first")).reset_index()
print(f"subject codes: {len(g)}  |  PD: {(g.status==1).sum()}  |  control: {(g.status==0).sum()}")
print(f"sessions: {meta.session.unique()}")
g.head(10)

subject codes: 32  |  PD: 24  |  control: 8
sessions: ['R01']


,subject,n,status
0,S01,6,1
1,S02,6,1
2,S04,6,1
3,S05,6,1
4,S06,6,1
5,S07,6,0
6,S08,6,1
7,S10,6,0
8,S13,6,0
9,S16,6,1


## 3. Corrected internal benchmark

Repeated stratified group nested cross-validation: 10 repeats x 5 outer x 3 inner.
The subject is the grouping unit in **both** loops, and predictions are averaged
per participant before any metric is computed.

Set `N_REPEATS = 3` for a faster check; the published figures use 10.

In [5]:
from pdpipe.models import build_models
from pdpipe.cv import run_nested_cv
from pdpipe.stats import summarise_models

N_REPEATS = 10   # 3 for a quick check
SEED = 20260917

res22 = run_nested_cv(ox, build_models(), n_repeats=N_REPEATS, base_seed=SEED)
summary = summarise_models(res22.fold_scores)
print(summary.round(3).to_string(index=False))

  repeat 1/10 done
  repeat 2/10 done
  repeat 3/10 done
  repeat 4/10 done
  repeat 5/10 done
  repeat 6/10 done
  repeat 7/10 done
  repeat 8/10 done
  repeat 9/10 done
  repeat 10/10 done
             model  mean  sd_fold  sd_repeat  ci_lo  ci_hi  min_fold  max_fold  n_folds  n_folds_undefined
LogisticRegression 0.856    0.170      0.042  0.832  0.880      0.40       1.0       50                  0
      RandomForest 0.850    0.176      0.070  0.809  0.892      0.20       1.0       50                  0
               KNN 0.828    0.222      0.080  0.782  0.877      0.00       1.0       50                  0
           XGBoost 0.811    0.198      0.050  0.783  0.841      0.40       1.0       50                  0
               SVM 0.796    0.207      0.063  0.758  0.832      0.30       1.0       50                  0
  GradientBoosting 0.788    0.213      0.060  0.756  0.824      0.20       1.0       50                  0
          CatBoost 0.761    0.227      0.081  0.713  0.807  

### Fold stability

This is why repetition matters: many outer test folds contain a single control
subject, and individual fold AUCs span the full range.

In [6]:
fc = res22.fold_composition
ot = fc[fc.split == "outer_test"]
print("control subjects per outer test fold:")
print(ot.n_hc_subjects.value_counts().sort_index().to_string())
fs = res22.fold_scores
print(f"\nfold AUC range: {fs.auc_subject.min():.2f} to {fs.auc_subject.max():.2f}")
print(f"model-fold estimates at or below 0.5: {(fs.auc_subject<=0.5).sum()} of {len(fs)}")

control subjects per outer test fold:
n_hc_subjects
1    20
2    30

fold AUC range: 0.00 to 1.00
model-fold estimates at or below 0.5: 59 of 450


## 4. Statistical comparison

A Wilcoxon test over five folds has almost no power. With 50 folds, plus effect
sizes, repeat-level bootstrap intervals and Holm correction across all 36
pairwise comparisons, the picture changes.

In [7]:
from pdpipe.stats import pairwise_comparisons

pc = pairwise_comparisons(res22.fold_scores)
lr = pc[(pc.model_a=="LogisticRegression") | (pc.model_b=="LogisticRegression")]
print(lr[["model_a","model_b","mean_diff","ci_lo","ci_hi",
          "rank_biserial","p_holm"]].round(4).to_string(index=False))

           model_a            model_b  mean_diff  ci_lo   ci_hi  rank_biserial  p_holm
          CatBoost LogisticRegression     -0.095 -0.130 -0.0580        -0.7908  0.0068
          LightGBM LogisticRegression     -0.126 -0.166 -0.0740        -0.7724  0.0093
      DecisionTree LogisticRegression     -0.104 -0.160 -0.0445        -0.6326  0.0534
LogisticRegression                SVM      0.060  0.034  0.0840         0.6443  0.2187
  GradientBoosting LogisticRegression     -0.068 -0.102 -0.0360        -0.4688  0.5344
LogisticRegression            XGBoost      0.045  0.020  0.0700         0.4400  0.9433
               KNN LogisticRegression     -0.028 -0.057  0.0030        -0.1954  1.0000
LogisticRegression       RandomForest      0.006 -0.024  0.0410         0.1190  1.0000


## 5. External validation

Three threshold policies, reported separately. Policy A never sees an external
label and is the only genuinely blind result.

In [8]:
from pdpipe.external import run_external, distribution_shift

ox16 = harmonised_oxford(ox)
res16 = run_nested_cv(ox16, build_models(), n_repeats=N_REPEATS, base_seed=SEED)

sel = {k:v for k,v in build_models().items()
       if k in ("LogisticRegression","RandomForest","CatBoost")}
ext = run_external(ox16, ist, sel, res16.oof_subject)
cols = ["model","policy","auc","balanced_accuracy","sensitivity","specificity","mcc"]
print(ext["table"][cols].round(3).to_string(index=False))

  repeat 1/10 done
  repeat 2/10 done
  repeat 3/10 done
  repeat 4/10 done
  repeat 5/10 done
  repeat 6/10 done
  repeat 7/10 done
  repeat 8/10 done
  repeat 9/10 done
  repeat 10/10 done
             model                policy   auc  balanced_accuracy  sensitivity  specificity   mcc
LogisticRegression        A_oxford_fixed 0.683              0.500        1.000        0.000 0.000
LogisticRegression   A_oxford_fixed_full 0.685              0.500        1.000        0.000 0.000
LogisticRegression B_istanbul_calibrated 0.683              0.652        0.553        0.750 0.264
LogisticRegression    C_in_sample_biased 0.685              0.664        0.500        0.828 0.290
      RandomForest        A_oxford_fixed 0.692              0.537        0.074        1.000 0.142
      RandomForest   A_oxford_fixed_full 0.706              0.548        0.096        1.000 0.162
      RandomForest B_istanbul_calibrated 0.692              0.649        0.766        0.531 0.280
      RandomForest    C_i

### Why transfer fails

Restricted to control participants, so disease status cannot explain the gap.
A feature differing by several standard deviations between cohorts among healthy
people reflects the extraction pipeline, not the illness.

In [9]:
ds = distribution_shift(ox16, ist)
print(ds[["feature","istanbul_column","oxford_mean","istanbul_mean",
          "smd_controls"]].head(8).round(4).to_string(index=False))
print(f"\nfeatures with |SMD| > 1 among controls: {(ds.abs_smd_controls>1).sum()} of 16")

       feature istanbul_column  oxford_mean  istanbul_mean  smd_controls
           PPE             PPE       0.2066         0.7463       -5.5034
      MDVP:RAP       rapJitter       0.0033         0.0006        1.9810
    Jitter:DDP       ddpJitter       0.0099         0.0018        1.9810
      MDVP:PPQ      ppq5Jitter       0.0034         0.0012        1.7517
MDVP:Jitter(%)    locPctJitter       0.0062         0.0023        1.4469
      MDVP:APQ    apq11Shimmer       0.0241         0.0554       -1.4263
  Shimmer:APQ5     apq5Shimmer       0.0179         0.0412       -1.2870
  MDVP:Shimmer      locShimmer       0.0297         0.0675       -1.2733

features with |SMD| > 1 among controls: 12 of 16


## 6. Computational cost — measured on THIS machine

Run this on the machine you will name in the paper. It prints an environment
report and writes a LaTeX block to `results/timing_table.tex`.

In [10]:
import json, os
os.makedirs("results", exist_ok=True)
from pdpipe.timing import benchmark, environment_report

table, env = benchmark(ox, build_models(), n_repetitions=5)
print("=== Environment ===")
for k, v in env.items():
    print(f"  {k}: {v}")
print("\n=== Timings ===")
print(table[["model","end_to_end_tuning_s","fit_mean_s",
             "inference_mean_ms","peak_rss_mb"]].to_string(index=False))

table.to_csv("results/timing.csv", index=False)
with open("results/environment.json","w") as f:
    json.dump(env, f, indent=2)

=== Environment ===
  cpu_model: Intel(R) Xeon(R) CPU @ 2.20GHz
  logical_cores: 2
  memory_gb: 12.67
  platform: Linux-6.6.122+-x86_64-with-glibc2.39
  python: 3.13.15
  numpy: 2.1.3
  pandas: 2.2.3
  scikit_learn: 1.6.1
  catboost: 1.2.10
  lightgbm: 4.6.0
  xgboost: 3.4.1
  thread_env: {'OMP_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1', 'NUMEXPR_NUM_THREADS': '1', 'VECLIB_MAXIMUM_THREADS': '1'}

=== Timings ===
             model  end_to_end_tuning_s  fit_mean_s  inference_mean_ms  peak_rss_mb
LogisticRegression               0.2115      0.0068              1.747        388.3
               SVM               0.2481      0.0106              3.606        388.3
               KNN               0.2206      0.0038              2.496        388.3
      DecisionTree               0.1977      0.0100              3.202        388.3
      RandomForest               8.0309      0.5242             28.078        388.3
  GradientBoosting               9.3348      0.2456 

## 7. Out-of-fold explainability

Explanations are recomputed on every held-out fold, using the hyperparameters
that fold selected, and aggregated with stability measures rather than read off
a single split.

In [11]:
from pdpipe.explain import run_explanations, stability_table

e = run_explanations(ox, build_models(), res22.best_params,
                     model_names=["CatBoost","RandomForest"],
                     n_repeats=min(5, N_REPEATS))
for m in ("CatBoost","RandomForest"):
    print(f"\n=== {m}: mean |SHAP| across folds ===")
    print(stability_table(e, m, "mean_abs_shap").head(6).round(4).to_string())


=== CatBoost: mean |SHAP| across folds ===
                mean      sd    p2.5   p97.5  median_rank  top_k_frequency
feature                                                                   
PPE           0.7460  0.2696  0.1979  1.2635          1.0             0.88
spread2       0.6804  0.3370  0.1867  1.4086          3.0             0.84
MDVP:Fo(Hz)   0.6683  0.3598  0.3042  1.4312          3.0             0.88
spread1       0.5728  0.2332  0.2019  1.1487          4.0             0.84
MDVP:Fhi(Hz)  0.4434  0.2123  0.1411  0.9033          5.0             0.52
DFA           0.3492  0.2597  0.0293  0.8493          6.0             0.32

=== RandomForest: mean |SHAP| across folds ===
                mean      sd    p2.5   p97.5  median_rank  top_k_frequency
feature                                                                   
PPE           0.0508  0.0114  0.0345  0.0777          1.0             1.00
spread1       0.0487  0.0112  0.0318  0.0736          2.0             1.00
spread2 

## 8. Save everything

In [12]:
for name, df in [("main22_fold_scores", res22.fold_scores),
                 ("main22_fold_composition", res22.fold_composition),
                 ("main22_best_params", res22.best_params),
                 ("main22_pairwise", pc),
                 ("main16_fold_scores", res16.fold_scores),
                 ("external", ext["table"]),
                 ("harmonisation_shift", ds),
                 ("explanations", e)]:
    df.to_csv(f"results/{name}.csv", index=False)
print("written to results/")

# On Colab, download everything as one archive:
# !zip -qr results.zip results && from google.colab import files; files.download("results.zip")

written to results/
